# Building Convolutional Neural Network (CNN) for MNIST using TensorFlow Keras

This notebook contains the complete code from the presentation.

## 1. Import Required Libraries

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
import os
from distutils.version import LooseVersion as Version

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

TensorFlow version: 2.21.0
NumPy version: 2.5.3


## 2. 1D Convolution from Scratch

In [3]:
def conv1d(x, w, p=0, s=1):
    w_rot = np.array(w[::-1])
    x_padded = np.array(x)
    if p > 0:
        zero_pad = np.zeros(shape=p)
        x_padded = np.concatenate([zero_pad, x_padded, zero_pad])
    res = []
    for i in range(0, int((len(x_padded) - len(w_rot)) / s) + 1, s):
        res.append(np.sum(x_padded[i:i + w_rot.shape[0]] * w_rot))
    return np.array(res)

# Test
x = [1, 3, 2, 4, 5, 6, 1, 3]
w = [1, 0, 3, 1, 2]

print('Conv1d rendered:', conv1d(x, w, p=2, s=1))
print('NumPy Result:   ', np.convolve(x, w, mode='same'))

Conv1d rendered: [ 5. 14. 16. 26. 24. 34. 19. 22.]
NumPy Result:    [ 5 14 16 26 24 34 19 22]


## 3. 2D Convolution from Scratch

In [5]:
def conv2d(X, W, p=(0, 0), s=(1, 1)):
    W_rot = np.array(W[::-1, ::-1])
    X_orig = np.array(X)
    n1 = X_orig.shape[0] + 2 * p[0]
    n2 = X_orig.shape[1] + 2 * p[1]
    X_padded = np.zeros(shape=(n1, n2))
    X_padded[p[0]:p[0] + X_orig.shape[0],
             p[1]:p[1] + X_orig.shape[1]] = X_orig
    res = []
    for i in range(0, int((X_padded.shape[0] - W_rot.shape[0]) / s[0]) + 1, s[0]):
        res.append([])
        for j in range(0, int((X_padded.shape[1] - W_rot.shape[1]) / s[1]) + 1, s[1]):
            X_sub = X_padded[i:i + W_rot.shape[0],
                             j:j + W_rot.shape[1]]
            res[-1].append(np.sum(X_sub * W_rot))
    return np.array(res)

# Test
X = [[1, 3, 2, 4],
     [5, 6, 1, 3],
     [1, 2, 0, 2],
     [3, 4, 3, 2]]

W = [[1, 0, 3],
     [1, 2, 1],
     [0, 1, 1]]

def conv2d(X, W, p=(0, 0), s=(1, 1)):
    W_rot = np.array(W)[::-1, ::-1]
    X_orig = np.array(X)

    n1 = X_orig.shape[0] + 2 * p[0]
    n2 = X_orig.shape[1] + 2 * p[1]
    X_padded = np.zeros(shape=(n1, n2))

    X_padded[p[0]:p[0] + X_orig.shape[0],
             p[1]:p[1] + X_orig.shape[1]] = X_orig

    res = []
    for i in range(0, X_padded.shape[0] - W_rot.shape[0] + 1, s[0]):
        res.append([])
        for j in range(0, X_padded.shape[1] - W_rot.shape[1] + 1, s[1]):
            X_sub = X_padded[i:i + W_rot.shape[0],
                             j:j + W_rot.shape[1]]
            res[-1].append(np.sum(X_sub * W_rot))

    return np.array(res)

print('Rendering Conv2d:\n', conv2d(X, W, p=(1, 1), s=(1, 1)))
print('\nSciPy result:\n', __import__('scipy').signal.convolve2d(X, W, mode='same'))
print('\nSciPy result:\n', __import__('scipy').signal.convolve2d(X, W, mode='same'))

Rendering Conv2d:
 [[11. 25. 32. 13.]
 [19. 25. 24. 13.]
 [13. 28. 25. 17.]
 [11. 17. 14.  9.]]

SciPy result:
 [[11 25 32 13]
 [19 25 24 13]
 [13 28 25 17]
 [11 17 14  9]]

SciPy result:
 [[11 25 32 13]
 [19 25 24 13]
 [13 28 25 17]
 [11 17 14  9]]


## 4. Loss Functions Examples

In [7]:
# Binary Cross Entropy
bce_probas = tf.keras.losses.BinaryCrossentropy(from_logits=False)
bce_logits = tf.keras.losses.BinaryCrossentropy(from_logits=True)

logits = tf.constant([0.8])
probas = tf.keras.activations.sigmoid(logits)

y_true = tf.constant([1.0])

print('BCE (probability): {:.4f}'.format(
    bce_probas(y_true=y_true, y_pred=probas).numpy()),
    '(Logit): {:.4f}'.format(
    bce_logits(y_true=y_true, y_pred=logits).numpy()))

BCE (probability): 0.3711 (Logit): 0.3711


In [9]:
# Categorical Cross Entropy
cce_probas = tf.keras.losses.CategoricalCrossentropy(from_logits=False)
cce_logits = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

logits = tf.constant([[1.5, 0.8, 2.1]])
probas = tf.keras.activations.softmax(logits)

y_true_categorical = tf.constant([[0.0, 0.0, 1.0]])

print('CCE (probability): {:.4f}'.format(
    cce_probas(y_true=y_true_categorical, y_pred=probas).numpy()),
    '(Logit): {:.4f}'.format(
    cce_logits(y_true=y_true_categorical, y_pred=logits).numpy()))

CCE (probability): 0.5996 (Logit): 0.5996


In [11]:
# Sparse Categorical Cross Entropy
sp_cce_probas = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
sp_cce_logits = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

sparse_label = tf.constant([2], dtype=tf.int32)

print('Sparse CCE (probability): {:.4f}'.format(
    sp_cce_probas(y_true=sparse_label, y_pred=probas).numpy()),
    '(Logit): {:.4f}'.format(
    sp_cce_logits(y_true=sparse_label, y_pred=logits).numpy()))

Sparse CCE (probability): 0.5996 (Logit): 0.5996


## 5. Loading MNIST Dataset

In [ ]:
mnist_bldr = tfds.builder('mnist')
%pip install -q importlib_resources
mnist_bldr.download_and_prepare()
datasets = mnist_bldr.as_dataset(shuffle_files=False)

print(datasets.keys())
mnist_train_orig, mnist_test_orig = datasets['train'], datasets['test']

ModuleNotFoundError: No module named 'importlib_resources'

## 6. Preprocessing & Creating Validation Set

In [ ]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
NUM_EPOCHS = 20

mnist_train = mnist_train_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32)/255.0,
                  tf.cast(item['label'], tf.int32)))

mnist_test = mnist_test_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32)/255.0,
                  tf.cast(item['label'], tf.int32)))

tf.random.set_seed(1)
mnist_train = mnist_train.shuffle(
    buffer_size=BUFFER_SIZE,
    reshuffle_each_iteration=False)

mnist_valid = mnist_train.take(10000).batch(BATCH_SIZE)
mnist_train = mnist_train.skip(10000).batch(BATCH_SIZE)

## 7. Building the CNN Model

In [ ]:
model = tf.keras.Sequential()

model.add(tf.keras.layers.Conv2D(
    filters=32, kernel_size=(5, 5),
    strides=(1, 1), padding='same',
    data_format='channels_last',
    name='conv_1', activation='relu'))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2, 2), name='pool_1'))

model.add(tf.keras.layers.Conv2D(
    filters=64, kernel_size=(5, 5),
    strides=(1, 1), padding='same',
    name='conv_2', activation='relu'))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2, 2), name='pool_2'))

In [ ]:
# Check output shape after Conv + Pool
print(model.compute_output_shape(input_shape=(16, 28, 28, 1)))

In [ ]:
# Add Flatten layer
model.add(tf.keras.layers.Flatten())
print(model.compute_output_shape(input_shape=(16, 28, 28, 1)))

In [ ]:
# Add Dense + Dropout + Output layer
model.add(tf.keras.layers.Dense(
    units=1024, name='fc_1',
    activation='relu'))

model.add(tf.keras.layers.Dropout(rate=0.5))

model.add(tf.keras.layers.Dense(
    units=10, name='fc_2',
    activation='softmax'))

## 8. Build and Compile the Model

In [ ]:
tf.random.set_seed(1)
model.build(input_shape=(None, 28, 28, 1))

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'])

model.summary()

## 9. Train the Model

In [ ]:
history = model.fit(mnist_train,
                    epochs=NUM_EPOCHS,
                    validation_data=mnist_valid,
                    shuffle=True)

## 10. Plot Learning Curves

In [ ]:
hist = history.history
x_arr = np.arange(len(hist['loss'])) + 1

fig = plt.figure(figsize=(12, 4))

ax = fig.add_subplot(1, 2, 1)
ax.plot(x_arr, hist['loss'], '-o', label='Train loss')
ax.plot(x_arr, hist['val_loss'], '--<', label='Validation loss')
ax.set_xlabel('Epoch', size=15)
ax.set_ylabel('Loss', size=15)
ax.legend(fontsize=15)

ax = fig.add_subplot(1, 2, 2)
ax.plot(x_arr, hist['accuracy'], '-o', label='Train acc.')
ax.plot(x_arr, hist['val_accuracy'], '--<', label='Validation acc.')
ax.set_xlabel('Epoch', size=15)
ax.set_ylabel('Accuracy', size=15)
ax.legend(fontsize=15)

plt.show()

## 11. Evaluate on Test Set

In [ ]:
test_results = model.evaluate(mnist_test.batch(20))
print('\nTest Accuracy: {:.2f}%'.format(test_results[1] * 100))

## 12. Make Predictions and Visualize

In [ ]:
batch_test = next(iter(mnist_test.batch(12)))
preds = model(batch_test[0])

print("Predictions shape:", preds.shape)
preds = tf.argmax(preds, axis=1)
print("Predicted labels:", preds.numpy())

fig = plt.figure(figsize=(12, 4))
for i in range(12):
    ax = fig.add_subplot(2, 6, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    img = batch_test[0][i, :, :, 0]
    ax.imshow(img, cmap='gray_r')
    ax.text(0.9, 0.1, '{}'.format(preds[i]),
            size=15, color='blue',
            horizontalalignment='center',
            verticalalignment='center',
            transform=ax.transAxes)

plt.show()

## 13. Save the Model

In [ ]:
if not os.path.exists('models'):
    os.mkdir('models')

model.save('models/mnist-cnn.h5')
print("Model saved successfully as 'models/mnist-cnn.h5'")